保险场景 VLM使用

In [1]:
import os
from openai import OpenAI
import pandas as pd
client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"), 
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

# 调用VLM，得到推理结果
# user_prompt：用户想要分析的内容
# image_url：想要分析的图片
def get_response(user_prompt, image_url):
    # 得到image_url_list，一张图片也放到[]中
    if image_url.startswith('[') and ',' in image_url:
        # 属于image_url list
        image_url = image_url.strip()
        image_url = image_url[1:-1]
        image_url_list = image_url.split(',')
        image_url_list = [temp_url.strip() for temp_url in image_url_list]
    else:
        image_url_list = [image_url]
    # 得到messages
    content = [{"type": "text", "text": f"{user_prompt}"}]
    for temp_url in image_url_list: 
        image_url = f"https://vl-image.oss-cn-shanghai.aliyuncs.com/{temp_url}.jpg"
        content.append({"type": "image_url","image_url": {"url": f"{image_url}"}})
    messages=[{
                "role": "user",
                "content": content
            }
        ]

    print(f'messages={messages}')
    completion = client.chat.completions.create(
        model="qwen-vl-max", #qwen-vl-plus
        messages=messages    
        )
    #print(completion.model_dump_json())
    return completion

In [2]:
df = pd.read_excel('./prompt_template_cn.xlsx')
df['response'] = ''
for index, row in df.iterrows():
    user_prompt = row['prompt']
    image_url = row['image']
    # 得到VLM推理结果
    completion = get_response(user_prompt, image_url)
    response = completion.choices[0].message.content
    df.loc[index, 'response'] = response
    print(f"{index+1} {user_prompt} {image_url}")
df.to_excel('./prompt_template_cn_result.xlsx', index=False)

messages=[{'role': 'user', 'content': [{'type': 'text', 'text': '假设你是一位健康保险专家。这里有一份保单持有人提供的医疗文件的图片。请从中提取详细信息。'}, {'type': 'image_url', 'image_url': {'url': 'https://vl-image.oss-cn-shanghai.aliyuncs.com/1-Chinese-document-extraction.jpg'}}]}]
1 假设你是一位健康保险专家。这里有一份保单持有人提供的医疗文件的图片。请从中提取详细信息。 1-Chinese-document-extraction
messages=[{'role': 'user', 'content': [{'type': 'text', 'text': '假设你是一位健康保险专家。这里有一份保单持有人提供的医疗文件的图片。请从中提取详细信息。'}, {'type': 'image_url', 'image_url': {'url': 'https://vl-image.oss-cn-shanghai.aliyuncs.com/2-Japanese-document-extraction.jpg'}}]}]
2 假设你是一位健康保险专家。这里有一份保单持有人提供的医疗文件的图片。请从中提取详细信息。 2-Japanese-document-extraction
messages=[{'role': 'user', 'content': [{'type': 'text', 'text': '假设你是一位健康保险专家。这里有一份保单持有人提供的医疗文件的图片。请从中提取详细信息。'}, {'type': 'image_url', 'image_url': {'url': 'https://vl-image.oss-cn-shanghai.aliyuncs.com/3-French-document-extraction.jpg'}}]}]
3 假设你是一位健康保险专家。这里有一份保单持有人提供的医疗文件的图片。请从中提取详细信息。 3-French-document-extraction
messages=[{'role': 'user', 'content': [{

In [3]:
df

,id,prompt,image,response
0,1,假设你是一位健康保险专家。这里有一份保单持有人提供的医疗文件的图片。请从中提取详细信息。,1-Chinese-document-extraction,根据您提供的医疗文件图片，以下是提取的详细信息：\n\n---\n\n### **基本信息*...
1,2,假设你是一位健康保险专家。这里有一份保单持有人提供的医疗文件的图片。请从中提取详细信息。,2-Japanese-document-extraction,很抱歉，我无法查看或分析图片内容。您提供的信息是一份诊断书的模板，但其中的关键字段（如姓名、...
2,3,假设你是一位健康保险专家。这里有一份保单持有人提供的医疗文件的图片。请从中提取详细信息。,3-French-document-extraction,作为健康保险专家，我将从您提供的医疗报告中提取关键信息，并以结构化方式整理，以便用于保险理赔...
3,4,假设你是一位健康保险专家。这里有一份保单持有人提供的医疗文件的图片。请从中提取详细信息。,4-German-document-extraction,作为健康保险专家，我将从您提供的医疗文件图片中提取关键信息，并进行结构化整理，以便用于保险评...
4,5,假设你是一位健康保险专家。这里有一份保单持有人提供的医疗文件的图片。请从中提取详细信息。,5-Korean-document-extraction,作为健康保险专家，我将从您提供的医疗文件（《上解诊断书》）中提取并解析关键信息。该文件为韩文...
